[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AGRIdigital-Climbat/PlantillaMapa/blob/main/07_NDVI_Fenologia/01_ndvi_fenologia.ipynb)

# NDVI y fenologia: ciclo vegetativo vs. calendario agricola
## PROSUCO - CLIMBAT | Analisis por municipio de Bolivia (GEE + Colab)

> **Area de estudio = tu GeoJSON.** Este notebook usa por defecto:
>
> `RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"` (compartido `Cualquiera con el enlace`).
> El GeoJSON **siempre manda**: al cargarse, `roi`, `ZONA['bbox']`, `ZONA['centro']`
> y `area_km2` se recalculan desde el poligono. El `bbox` de CONFIG solo es respaldo
> si cambias a `AOI_MODE = "bbox"`.

**Orden de ejecucion:** 1) Instalacion -> 2) Autenticacion GEE (`EE_PROJECT`) ->
3) CONFIG (anos + satelite + umbrales) -> 4) AOI desde Drive -> 5) celdas del modulo.
Solo edita la celda **CONFIGURACION MAESTRA**.


In [ ]:
# ============================================================
# 1.1 Instalación de librerías (ejecutar una sola vez en Colab)
# ============================================================

import importlib
import subprocess
import sys


def _asegurar(paquete, modulo=None):
    """Instala el paquete solo si no está disponible."""
    try:
        importlib.import_module(modulo or paquete)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", paquete]
        )


_asegurar("earthengine-api", "ee")
_asegurar("geemap")
_asegurar("geopandas")
_asegurar("pandas")
_asegurar("numpy")
_asegurar("matplotlib")

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

print("Librerías listas.")
print("earthengine-api:", ee.__version__)
print("geemap:", geemap.__version__)

In [ ]:
# ============================================================
# 1.2 Autenticación en Google Earth Engine
# ============================================================
# >>> REEMPLAZAR por el id del proyecto de Cloud de PROSUCO/CLIMBAT <<<

EE_PROJECT = "iotapp7"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

print("Earth Engine inicializado correctamente.")
print("Proyecto:", EE_PROJECT)

## Configuración maestra

> **✏️ Esta es la única celda que se edita para diagnosticar una zona nueva.**

| Bloque | Qué define | Verificación previa |
|---|---|---|
| `ZONA` | nombre, bbox `[oeste, sur, este, norte]` y punto de vista | dibujar en [geojson.io](https://geojson.io) o en el editor GEE y copiar coordenadas |
| `AOI_MODE`/`RUTA_AOI` | fuente del AOI: `"bbox"` o `"auto"` (polígono **GeoJSON**) | exportar el polígono desde QGIS como **GeoJSON**; en `RUTA_AOI` se puede pegar un **enlace de Google Drive** (el notebook lo descarga solo) o una ruta de Drive |
| `PUNTOS_LOCALES` | manantiales/bofedales del saber local | levantamiento con las comunidades |
| `FECHA_INICIO / FECHA_FIN` | ventana analítica (**30+ años**) | default 1996-01-01 → 2026-12-31 |
| `CALENDARIO` | meses de siembra/cosecha y estación seca | observadores Yapuchiri |
| `UMBRALES` | todos los cortes analíticos en un solo lugar | criterio técnico + experiencia local |
| `ESCALAS` | resolución de cómputo (m); subir si la zona es grande | — |

In [ ]:
# ===========================================================================
# CONFIGURACIÓN MAESTRA — ÚNICA CELDA QUE SE EDITA PARA CADA ZONA
# ===========================================================================

# --- 1. Zona de intervención ------------------------------------------------
# Fuente del AOI:
#   "bbox" -> usa el rectángulo ZONA["bbox"] de abajo (rápido, sin Drive)
#   "auto" -> usa un polígono .geojson. RUTA_AOI admite 3 formas:
#       * Enlace o ID de Google Drive  -> se DESCARGA solo (no subes nada a mano)
#           "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"
#           "11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N"
#       * Ruta en tu Drive ya montado  -> se lee directo, SIN copiar a /content
#           "/content/drive/MyDrive/CLIMBAT/zona.geojson"
#       * Carpeta en Drive             -> toma el único .geojson que contenga
#           "/content/drive/MyDrive/CLIMBAT"
AOI_MODE = "auto"
RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"

ZONA = {
    # Coordenadas geocodificadas: Escoma (La Paz, Bolivia) ~3832 msnm
    # (solo se usan como respaldo cuando AOI_MODE = "bbox")
    "nombre": "Escoma, La Paz (prueba)",
    "bbox": [-69.33, -15.86, -68.93, -15.46],   # [oeste, sur, este, norte]
    "centro": [-69.1289, -15.6615],             # punto de vista del mapa
    "zoom": 10,
}

# --- 2. Puntos levantados con comunidades (saber local) ---------------------
# Listas de [longitud, latitud]. Dejar vacías si aún no hay levantamiento.
PUNTOS_LOCALES = {
    "manantiales": [],
    "bofedales": [],
}

# --- 3. Ventana analítica: 30+ años hacia atrás desde el año actual ---------
# Regla de la plantilla: 1996-01-01 -> 2026-12-31 (todo análisis de rango de años usa esta).
# Los datasets usan su período nativo si es mayor: CHIRPS 1981+ · ERA5-Land 1950+
# · JRC 1984–2021 · MODIS 2000+ (NDVI) / 2001+ (MCD12Q1 agricultura) · SMAP 2015+
FECHA_INICIO = "1996-01-01"
FECHA_FIN = "2026-12-31"

# --- 4. Calendario agrícola local (verificar con observadores Yapuchiri) ----
CALENDARIO = {
    "siembra": 11,       # mes de siembra (1–12)
    "cosecha": 4,        # mes de cosecha (1–12)
    "meses_estacion_seca": [5, 6, 7, 8, 9, 10],   # se usa en M10 (bofedales)
}

# --- 5. Umbrales analíticos (ajustar con criterio técnico y local) ----------
UMBRALES = {
    "helada_c": 0.0,               # °C mínima para contar un día de helada (M4)
    "helada_grave_c": -2.0,        # °C de helada grave (daño severo) (M4)
    "mes_seco_mm": 30.0,           # mm/mes para clasificar mes seco (M2)
    "spi_severo": -1.5,            # umbral de sequía severa SPI-3 (M5)
    "ndvi_anomalia_z": -1.5,       # anomalía NDVI severa (M7)
    "manantial_occurrencia": 100,  # % ocurrencia JRC mínima (M9)
    "manantial_drenaje_max": 50,   # celdas acumuladas máx. (cabeceras) (M9)
    "bofedal_altitud_min": 3000,   # msnm mínimo (M10)
    "bofedal_altitud_max": 5200,   # msnm máximo (M10)
    "bofedal_pendiente_max": 5.0,  # grados de pendiente máxima (M10)
    "bofedal_seasonality": 8,      # meses con agua JRC mínimos (M10)
    "bofedal_ndvi_seco": 0.25,     # NDVI mínimo en estación seca (M10)
    "clases_agricolas": [12, 14],  # LC_Type1 MCD12Q1: 12=cultivos, 14=mosaico (M11)
}

# --- 6. Escalas de cómputo (metros) -----------------------------------------
# Subir si la zona es muy grande (más rápido, menos detalle)
ESCALAS = {"clima": 5000, "modis": 250, "jrc": 90, "detalle": 30}
TILE_SCALE = 4

# ===========================================================================
# A partir de aquí no se edita nada
# ===========================================================================

MESES = list(range(1, 13))
MESES_ES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
            "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

roi = None   # se construye en la celda 1.4 (bbox o GeoJSON según AOI_MODE)
centro = None
ANIO_INI = int(FECHA_INICIO[:4])
ANIO_FIN = int(FECHA_FIN[:4])

# Años COMPLETOS (se excluye el año en curso de métricas y mapas anuales)
ANIOS_COMPLETOS = list(range(ANIO_INI, ANIO_FIN))
FILTRO_ANIOS_COMPLETOS = ee.Filter.calendarRange(
    ANIOS_COMPLETOS[0], ANIOS_COMPLETOS[-1], "year"
)

print("Zona:", ZONA["nombre"])
print("Ventana analítica:", FECHA_INICIO, "->", FECHA_FIN,
      f"({ANIO_FIN - ANIO_INI + 1} años calendario)")
print("Años completos para métricas:", ANIOS_COMPLETOS[0], "-", ANIOS_COMPLETOS[-1])
print("Calendario: siembra mes", CALENDARIO["siembra"],
      "| cosecha mes", CALENDARIO["cosecha"])
print("Umbrales activos:", UMBRALES)

In [ ]:
# ============================================================
# 1.3 Utilidades comunes (reutilizadas por todos los módulos)
# ============================================================

# Parámetros de reducción seguros contra "User memory limit exceeded":
#   bestEffort=True -> GEE sube la escala automáticamente si no cabe en memoria.
#   tileScale=8     -> divide el cálculo en tiles más pequeños (menos memoria c/u).
REDUCIR = dict(maxPixels=1e9, tileScale=8, bestEffort=True)
REDUCIR_AREA = dict(maxPixels=1e10, tileScale=16, bestEffort=True)


def apilar(imagenes, nombres):
    """Une imágenes de una banda en una sola imagen multibanda."""
    pila = None
    for img, nombre in zip(imagenes, nombres):
        banda = ee.Image(img).select(0).rename(nombre)
        pila = banda if pila is None else pila.addBands(banda)
    return pila


def reducir_bandas(imagenes, nombres, escala, tam=6, reductor=None, params=None):
    """reduceRegion de muchas bandas EN BLOQUES (una petición por bloque).

    Evita el error de memoria que ocurre al reducir de una sola vez una imagen
    con decenas de bandas (cada banda puede depender de cientos/miles de imágenes
    diarias). Devuelve un dict {nombre_banda: valor}.
    """
    reductor = reductor or ee.Reducer.mean()
    params = params or REDUCIR
    salida = {}
    for i in range(0, len(imagenes), tam):
        pila = apilar(imagenes[i:i + tam], nombres[i:i + tam])
        salida.update(
            pila.reduceRegion(reductor, roi, escala, **params).getInfo()
        )
    return salida


def composite_mensual(coleccion, anios, banda=None):
    """Reduce una colección diaria a UN compuesto por año-mes (una imagen c/u).

    Aligera ~30x la memoria de las climatologías de 30 años. `anios` debe listar
    solo años con dato (evita colecciones vacías). Con `banda`, deja una sola banda.
    """
    imgs = []
    for y in anios:
        for m in MESES:
            i0 = f"{y}-{m:02d}-01"
            i1 = (pd.Timestamp(i0) + pd.offsets.MonthBegin(1)).strftime("%Y-%m-%d")
            sub = coleccion.filterDate(i0, i1)
            if banda:
                sub = sub.select(banda)
            imgs.append(sub.mean().set("system:time_start", ee.Date(i0).millis()))
    return ee.ImageCollection(imgs)


def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente).

    Evita que el relleno de color tape el mapa y meta ruido en la interpretación;
    deja el límite del área bien visible por encima de las demás capas.
    """
    return roi.style(color=color, fillColor="00000000", width=ancho)


def promediar_aoi(imagen, banda, escala):
    """Promedio de una banda dentro del AOI -> valor Python (o None)."""
    d = (
        imagen.select(banda)
        .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        .getInfo()
    )
    return d.get(banda)


def serie_aoi(coleccion, banda, escala, bloque_anios=5):
    """Extrae el promedio del AOI por imagen, EN BLOQUES de años.

    Procesar la serie completa de una vez agota la memoria; por eso se trocea
    (por defecto cada 5 años) y se concatenan los resultados.
    """
    registros = []

    def _una_imagen(img):
        st = (
            img.select(banda)
            .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        )
        return ee.Feature(None, {
            "date": ee.Date(img.get("system:time_start")).format("yyyy-MM-dd"),
            "valor": st.get(banda),
        })

    for y0 in range(ANIO_INI, ANIO_FIN + 1, bloque_anios):
        y1 = min(y0 + bloque_anios - 1, ANIO_FIN)
        sub = coleccion.filterDate(f"{y0}-01-01", f"{y1 + 1}-01-01")
        fc = ee.FeatureCollection(sub.map(_una_imagen)).getInfo()
        registros += [f["properties"] for f in fc["features"]]
    return registros


print("Utilidades: apilar(), reducir_bandas(), composite_mensual(), "
      "promediar_aoi(), serie_aoi()")

In [ ]:
# ============================================================
# 1.4 Área de estudio (AOI) y mapa base
# ============================================================
# Fuente del AOI (ver CONFIG):
#   AOI_MODE = "bbox" -> rectángulo ZONA["bbox"].
#   AOI_MODE = "auto" -> polígono desde un archivo .geojson en Google Drive.
#                        (En QGIS: Exportar -> Guardar entidades como -> GeoJSON,
#                         CRS EPSG:4326; subir a Drive y ajustar RUTA_AOI.)

def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente)."""
    # ee.Geometry NO tiene .style() en la API de Python -> se envuelve en FeatureCollection
    fc = ee.FeatureCollection([ee.Feature(roi)])
    return fc.style(color=color, fillColor="00000000", width=ancho)


if AOI_MODE == "auto":
    from google.colab import drive
    drive.mount("/content/drive")

    import os
    import re
    import glob
    import geopandas as gpd

    ruta = str(RUTA_AOI).strip()

    # (a) Enlace o ID de Google Drive -> descarga automática (sin subir nada a mano).
    #     El archivo debe estar compartido como "Cualquiera con el enlace".
    if ruta.startswith("http") or ("/" not in ruta and len(ruta) >= 20):
        _asegurar("gdown")
        import gdown

        m = re.search(r"(?:/d/|id=)([-\w]{20,})", ruta)
        fid = m.group(1) if m else ruta
        ext = (
            ".zip" if ruta.lower().endswith(".zip") else
            ".kml" if ruta.lower().endswith(".kml") else
            ".gpkg" if ruta.lower().endswith(".gpkg") else
            ".geojson"
        )
        destino = "/content/zona_aoi" + ext
        print("Descargando AOI desde Google Drive...")
        gdown.download(id=fid, output=destino, quiet=False)
        ruta = destino
    # (b) Carpeta en Drive -> usar el único .geojson que contenga
    elif os.path.isdir(ruta):
        cands = sorted(glob.glob(os.path.join(ruta, "*.geojson")))
        if not cands:
            raise FileNotFoundError(f"No se encontró ningún .geojson en {ruta}")
        ruta = cands[0]
    # (c) Ruta de archivo en el Drive montado -> se lee directo (sin copiar)

    gdf = gpd.read_file(ruta).to_crs(4326)
    try:
        geometria = gdf.geometry.union_all()      # shapely >= 2 (disuelve features)
    except AttributeError:
        geometria = gdf.geometry.unary_union      # shapely < 2
    gdf = gpd.GeoDataFrame(geometry=[geometria], crs=4326)

    roi = geemap.geopandas_to_ee(gdf).geometry()

    minx, miny, maxx, maxy = geometria.bounds
    ZONA["bbox"] = [minx, miny, maxx, maxy]
    ZONA["centro"] = [(minx + maxx) / 2, (miny + maxy) / 2]
    ZONA["zoom"] = 10
    print("AOI cargado desde:", ruta)
else:
    roi = ee.Geometry.Rectangle(ZONA["bbox"], proj="EPSG:4326", geodesic=False)

centro = ee.Geometry.Point(ZONA["centro"])
area_km2 = roi.area(maxError=1e6).divide(1e6).getInfo()

Map_aoi = geemap.Map()
Map_aoi.centerObject(roi, ZONA["zoom"])
Map_aoi.add_basemap("HYBRID")
Map_aoi.addLayer(borde_aoi(), {}, f"AOI: {ZONA['nombre']}")

# Puntos del saber local (si se cargaron en CONFIG)
for tipo, color in (("manantiales", "cyan"), ("bofedales", "magenta")):
    coords = PUNTOS_LOCALES.get(tipo, [])
    if coords:
        Map_aoi.addLayer(
            ee.Geometry.MultiPoint(coords),
            {"color": color},
            f"{tipo.capitalize()} (saber local)",
        )

print(f"Zona: {ZONA['nombre']}")
print(f"Área del AOI: {area_km2:,.1f} km²")
print(f"Extensión (bbox): {ZONA['bbox']}")
print("Puntos de saber local cargados:",
      {k: len(v) for k, v in PUNTOS_LOCALES.items()})

Map_aoi

# M7. NDVI y fenología: cuándo crece la vegetación y si calza con el calendario
**Dimensiones:** *temporalidad* (inicio/pico vs. siembra), *estacionalidad* (curva anual),
*intensidad* (NDVI pico), *tendencia* y *anomalías* (z-score anual).

* **Primario:** `MODIS/061/MOD13Q1` — NDVI (×0,0001) cada 16 días, 250 m, **2000–actual**
  (la ventana CONFIG 1996–2026 se aplica desde que el producto existe; 1996–1999 sin dato).
* **Alternativas:** Sentinel-2 SR (10 m, 2017+) para detalle intra-parcela; Landsat 8 (30 m,
  2013+) para series largas de 30 m.
* **Calidad:** se enmascaran píxeles con nube/sombra usando la banda `state_10000` (bits 1–2).

<details>
<summary><b>📐 NDVI: qué es y cómo leerlo</b> — clic para ver</summary>
<br>

NDVI = (NIR − Rojo) / (NIR + Rojo), escala **−1 a 1**. Valores guía:

| NDVI | Significado |
|---|---|
| < 0,1 | agua, nieve, roca desnuda |
| 0,1–0,25 | suelo desnudo, cultivo recién sembrado |
| 0,25–0,5 | cultivo en crecimiento |
| > 0,5 | vegetación densa/saludable |

La **curva anual** (climatología) muestra el ciclo: subida = inicio de crecimiento, pico = máximo
desarrollo, bajada = senescencia/cosecha. Compararla con el `CALENDARIO` de CONFIG es el corazón
de este módulo.

</details>

In [ ]:
# ============================================================
# M7.1 Serie NDVI y curva estacional (Sentinel-2)
# Extracción autocontenida — sin depender de serie_aoi
# ============================================================

# --- 1. Carga de la colección de Sentinel-2 (SR Harmonizado) -----------------
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate(FECHA_INICIO, FECHA_FIN)
    .filterBounds(roi)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
)

n_imagenes = s2.size().getInfo()
print(f"Imágenes Sentinel-2 en la ventana: {n_imagenes}")

if n_imagenes == 0:
    raise RuntimeError(
        f"No hay imágenes de Sentinel-2 entre {FECHA_INICIO} y {FECHA_FIN} "
        f"en el AOI. Revisa las fechas o el área de estudio."
    )

# --- 2. Función NDVI + máscara de nubes (usando la banda SCL) ----------------
def calcular_ndvi(img):
    """Calcula NDVI y enmascara nubes, sombras y nieve con la banda SCL."""
    scl = img.select('SCL')
    # SCL: 4=vegetación, 5=suelo desnudo, 6=agua, 7=no clasificado
    # Descartamos: 3=sombra de nube, 8=nube media, 9=nube alta, 10=cirros, 11=nieve
    mascara = (scl.neq(3)
               .And(scl.neq(8))
               .And(scl.neq(9))
               .And(scl.neq(10))
               .And(scl.neq(11)))
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('ndvi')
    return ndvi.updateMask(mascara).copyProperties(img, ['system:time_start'])

ndvi_col = s2.map(calcular_ndvi)

# --- 3. Extracción por bloques usando reduceRegion directamente ---------------
def extraer_ndvi(coleccion, aoi, escala=10, chunk=200):
    """Extrae la media de NDVI en el AOI para cada imagen, por bloques."""
    n = coleccion.size().getInfo()
    print(f"Extrayendo {n} imágenes en bloques de {chunk}...")
    filas = []
    lista = coleccion.toList(n)
    for i in range(0, n, chunk):
        sub = ee.ImageCollection(lista.slice(i, min(i + chunk, n)))
        def reducir(img):
            stats = ee.Image(img).reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=aoi,
                scale=escala,
                maxPixels=1e13,
                bestEffort=True,
                tileScale=8,
            )
            return ee.Feature(None, {
                'fecha': ee.Image(img).date().format('YYYY-MM-dd'),
                'valor': stats.get('ndvi'),
            })
        bloque = sub.map(reducir).getInfo()['features']
        for f in bloque:
            filas.append(f['properties'])
        print(f"  · bloque {i + chunk if i + chunk < n else n}/{n} procesado")
    return pd.DataFrame(filas)

df_ndvi = extraer_ndvi(ndvi_col, roi, escala=20)

if df_ndvi.empty:
    raise RuntimeError("No se extrajo ninguna observación de NDVI.")

# --- 4. Normalización del DataFrame ------------------------------------------
df_ndvi['fecha'] = pd.to_datetime(df_ndvi['fecha'])
df_ndvi = df_ndvi.dropna(subset=['valor']).set_index('fecha').sort_index()

print(f"\nObservaciones válidas: {len(df_ndvi)}")
print(f"Rango: {df_ndvi.index.min().date()} → {df_ndvi.index.max().date()}")
print(f"Años presentes: {sorted(df_ndvi.index.year.unique())}")

# --- 5. Serie agregada por mes (para gráfico legible) ------------------------
df_mensual = (
    df_ndvi
    .resample('MS')
    .agg({'valor': 'median'})
    .dropna()
)

# --- 6. Curva estacional (solo años completos) --------------------------------
completos = df_ndvi[df_ndvi.index.year.isin(ANIOS_COMPLETOS)]

if completos.empty:
    print("⚠ No hay datos para años en ANIOS_COMPLETOS. "
          "Usando todos los años disponibles.")
    completos = df_ndvi

clim_ndvi = completos.groupby(completos.index.month)['valor'].mean()

mes_pico_ndvi = int(clim_ndvi.idxmax())
ndvi_pico = float(clim_ndvi.max())
ventana_cre = list(clim_ndvi[clim_ndvi >= 0.5 * ndvi_pico].index)

print(f"\nNDVI pico: {ndvi_pico:.2f} en el mes {mes_pico_ndvi} "
      f"({MESES_ES[mes_pico_ndvi - 1]})")
print("Ventana de crecimiento (NDVI >= 50% del pico): meses",
      [MESES_ES[m - 1] for m in ventana_cre])
print("Calendario CONFIG: siembra mes", CALENDARIO["siembra"],
      "| cosecha mes", CALENDARIO["cosecha"])

# --- 7. Gráficos --------------------------------------------------------------
fig, ejes = plt.subplots(1, 2, figsize=(15, 4.8))

# Panel 1 — Serie temporal
ejes[0].plot(df_ndvi.index, df_ndvi['valor'], lw=0.4, alpha=0.4,
             color="#66bb6a", label="Observaciones (10 m)")
ejes[0].plot(df_mensual.index, df_mensual['valor'], lw=1.8,
             color="#1b5e20", label="Mediana mensual")
ejes[0].set_title("Serie NDVI — promedio del AOI (Sentinel-2)")
ejes[0].set_ylabel("NDVI")
ejes[0].legend(fontsize=9)
ejes[0].grid(alpha=0.3)

# Panel 2 — Curva estacional vs. calendario
ejes[1].plot(MESES, clim_ndvi.values, marker="o", lw=2, color="#2e7d32",
             label="NDVI medio")
ejes[1].axvline(CALENDARIO["siembra"], color="#1565c0", ls="--",
                label="siembra (CONFIG)")
ejes[1].axvline(CALENDARIO["cosecha"], color="#ff9800", ls="--",
                label="cosecha (CONFIG)")
ejes[1].fill_betweenx([0, 1],
                      min(CALENDARIO["meses_estacion_seca"]),
                      max(CALENDARIO["meses_estacion_seca"]),
                      color="#ffe082", alpha=0.5, label="estación seca")
ejes[1].set_title("Curva estacional vs. calendario agrícola")
ejes[1].set_xticks(MESES)
ejes[1].set_xticklabels(MESES_ES)
ejes[1].set_ylim(0, 1)
ejes[1].legend(fontsize=9)
ejes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# M7.2 Fenología anual, anomalías y mapa NDVI en estación seca
# ============================================================

# --- Estadísticos anuales (meses con dato; reducción en bloques) --------------
anios_feno = [y for y in anios_ndvi if y in ANIOS_COMPLETOS]

imgs_b, nombres_b = [], []
for y in anios_feno:
    for m in MESES:
        co_m = (
            ndvi
            .filter(ee.Filter.calendarRange(y, y, "year"))
            .filter(ee.Filter.calendarRange(m, m, "month"))
        )
        imgs_b.append(co_m.mean().rename(f"{y}_{m:02d}"))
        nombres_b.append(f"{y}_{m:02d}")
d_b = reducir_bandas(imgs_b, nombres_b, ESCALAS["modis"], tam=12)

filas_feno = []
for y in anios_feno:
    vals = {m: d_b.get(f"{y}_{m:02d}") for m in MESES}
    vals = {m: v for m, v in vals.items() if v is not None}
    if not vals:
        continue
    mes_max = max(vals, key=vals.get)
    filas_feno.append({
        "anio": y,
        "ndvi_med": np.mean(list(vals.values())),
        "ndvi_max": vals[mes_max],
        "mes_pico": mes_max,
        "n_meses_dato": len(vals),
    })

df_feno = pd.DataFrame(filas_feno)
df_feno["z"] = (df_feno["ndvi_med"] - df_feno["ndvi_med"].mean()) / df_feno["ndvi_med"].std()

print(f"Años fenológicos: {len(df_feno)} ({df_feno['anio'].min()}-{df_feno['anio'].max()})")
display(df_feno.round(2))

peor_anio_n = int(df_feno.loc[df_feno["z"].idxmin(), "anio"])
mejor_anio_n = int(df_feno.loc[df_feno["z"].idxmax(), "anio"])
print(f"Año con vegetación más débil (z mínimo): {peor_anio_n} "
      f"(z={df_feno['z'].min():+.2f})")
print(f"Año con vegetación más fuerte: {mejor_anio_n} (z={df_feno['z'].max():+.2f})")

# --- Gráfico de anomalías anuales ----------------------------------------------
fig, ax = plt.subplots(figsize=(13, 4.2))
colores = np.where(df_feno["z"] < UMBRALES["ndvi_anomalia_z"], "#d32f2f",
                   np.where(df_feno["z"] < 0, "#ffb74d", "#43a047"))
ax.bar(df_feno["anio"], df_feno["z"], color=colores)
ax.axhline(0, color="black", lw=0.8)
ax.axhline(UMBRALES["ndvi_anomalia_z"], color="#d32f2f", ls="--",
           label=f"anomalía severa (z = {UMBRALES['ndvi_anomalia_z']})")
ax.set_title("Anomalía anual de NDVI medio (z-score): años con estrés vegetal")
ax.set_ylabel("z-score")
ax.legend()
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

# --- Mapa: NDVI en estación seca + NDVI máximo anual promedio ----------------
meses_sc = sorted(CALENDARIO["meses_estacion_seca"])
ndvi_seco_map = (
    ndvi.filter(ee.Filter.calendarRange(meses_sc[0], meses_sc[-1], "month"))
    .mean()
    .rename("ndvi_seco")
)
ndvi_max_map = ndvi.reduce(ee.Reducer.max()).rename("ndvi_max")

vis_ndvi = {"min": 0, "max": 0.8,
            "palette": ["edf8fb", "ccece6", "99d8c9", "66c2a4", "2ca25f", "006d2c"]}

Map_m7 = geemap.Map()
Map_m7.centerObject(roi, ZONA["zoom"])
Map_m7.add_basemap("HYBRID")
Map_m7.addLayer(ndvi_seco_map, vis_ndvi,
                f"NDVI medio estación seca ({MESES_ES[meses_sc[0] - 1]}–{MESES_ES[meses_sc[-1] - 1]})")
Map_m7.addLayer(ndvi_max_map, vis_ndvi, "NDVI máximo anual (promedio 2000+)", False)
Map_m7.addLayer(borde_aoi(), {}, "AOI")
Map_m7.add_colorbar(vis_ndvi, label="NDVI",
                    orientation="horizontal", position="bottomright",
                    layer_name="NDVI")
Map_m7

## Interpretación de M7
* **Temporalidad:** si el NDVI sube *antes* de la siembra configurada, el cultivo aprovecha humedad
  residual; si sube mucho *después*, puede indicar siembra tardía o dependencia de lluvia tardía.
  El pico debe caer antes de la cosecha; si el pico se corre hacia cosecha, hay retraso fenológico.
* **Estación seca (mapa):** vegetación verde en seco = fuentes de agua permanente (riego,
  humedad freática, bofedales) — enlaza con M6 y M10.
* **Anomalías:** años con z ≤ −1,5 son candidatos a fallas de cultivo; contrastar con el SPI (M5)
  y con el relato de los observadores.

**Preguntas guía**
1. ¿El inicio de subida del NDVI coincide con lo que ustedes llaman "arranque del cultivo"?
2. ¿El año de menor NDVI (z) corresponde a una sequía recordada?
3. ¿Qué zonas se mantienen verdes en estación seca? ¿son riego, humedad o bofedales?

## Cierre: checklist del modulo

- [ ] Parte base ejecutada (instalacion, autenticacion GEE, CONFIG, AOI del GeoJSON).
- [ ] Mapa(s) revisados contra el borde rojo del AOI (el GeoJSON siempre manda).
- [ ] Serie/tabla leidas + interpretacion contrastada con saber local (Yapuchiri/PACHASOL).
- [ ] Periodo y dataset anotados junto a cada numero (p. ej. `CHIRPS 1981-2025`).

### Como cambiar de satelite o de anos (patron en todos los modulos)

```python
FECHA_INICIO = "1996-01-01"  # en la celda CONFIG: ventana analitica
FECHA_FIN = "2026-12-31"
# --- Primario (default) ---
precip = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
# --- Alternativas: comentar la primaria y descomentar la elegida ---
# precip = ee.ImageCollection("NASA/GPM_L3/IMERG_V06")
# Cada alternativa indica resolucion, periodo y para que sirve.
```
